# Random Forest — Scheme 1 (Variable Test) — GAMEEMO

> Run on **Google Colab**. Mount your Google Drive and adjust `folder_path` before executing.


In [ ]:
import os, numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Define training size (change per experiment step) ──
train_size = 0.95  # 95% training; remainder becomes test set

# ── Load GAMEEMO dataset ──
folder_path = '/content/drive/My Drive/EEG Datasets/GAMEEMO/Denoised EEG Data/'
subject_files = [[f"S{i:02}_G{j}_Denoised.csv" for j in range(1, 5)] for i in range(1, 29)]

data_list = []
for subject_file_list in subject_files:
    for file in subject_file_list:
        file_path = os.path.join(folder_path, file)
        temp_data = pd.read_csv(file_path)
        data_list.append(temp_data)

data = pd.concat(data_list, ignore_index=True)
data = data.sample(frac=0.10, random_state=42).reset_index(drop=True)

X = data.drop(columns=['Valence', 'Arousal']).values
valence = data['Valence'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

valence_enc = LabelEncoder()
y_valence = valence_enc.fit_transform(valence)

# ── Train/Test split (test size varies with train_size) ──
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_valence, test_size=1 - train_size, random_state=42)

# ────────────────────────────────────────────────────────────
# Random Forest — Scheme 1 (Variable Test) — GAMEEMO
# ────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, max_depth=None,
    min_samples_split=2, min_samples_leaf=1, random_state=42, n_jobs=-1)

# ── 5-Fold Cross-Validation on training data ──
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy with {int(train_size * 100)}% training data: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (SD)")

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
print(f"Held-out Test Accuracy with {int(train_size * 100)}% training data: {accuracy:.4f}")

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred, normalize='true')
plt.figure(figsize=(5, 4))
sns.heatmap(cm * 100, annot=True, fmt=".2f", cmap="Blues")
plt.title(f"Confusion Matrix ({int(train_size * 100)}% Training Data)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()